gathering data

In [8]:
#data gathering
import pandas as pd
import numpy as np

data = pd.read_csv('twitter_training.csv')
data.head()

#data cleaning
data=data.drop(columns= ['Borderlands','2401'])
data = data.rename(columns={'Positive':'Sentiment','im getting on borderlands and i will murder you all ,':'Tweets'})
data =  data[(data['Sentiment']=='Positive')|(data['Sentiment']=='Negative')|(data['Sentiment']=='Neutral')]
data = data.dropna()
data = data.drop_duplicates()

data.isnull().sum()
data.duplicated().sum()

#feature encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
data['Sentiment']=le.fit_transform(data['Sentiment'])

#create tokens
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer( oov_token='<oov>')
tokenizer.fit_on_texts(data['Tweets'])

#sequenize tokens
sequence= tokenizer.texts_to_sequences(data['Tweets'])

#padding
from keras.utils import pad_sequences

sequence=pad_sequences(sequence, padding='pre')

#split into x and y 
x=sequence
y=data['Sentiment'].values

#train test split
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)






In [ ]:
#create model without embedding layer
#from tensorflow.keras.layers import Input,Dense,Embedding,SimpleRNN
#from tensorflow.keras.models import Sequential
#model=Sequential()
#model.add(SimpleRNN(32,input_shape=(99,1),return_sequences=False))
#model.add(Dense(3,activation='softmax'))
#model.summary()

#compile model
#model.compile(optimizer='adam',metrics=['accuracy'],loss='sparse_categorical_crossentropy')

#fit model
#model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=10,batch_size=64)

c:\Users\91976\Desktop\Generative AI Course\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_3 (SimpleRNN)        │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,187 (4.64 KB)

 Trainable params: 1,187 (4.64 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 16s 18ms/step - accuracy: 0.3651 - loss: 1.0970 - val_accuracy: 0.3706 - val_loss: 1.0953
Epoch 2/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.3698 - loss: 1.0934 - val_accuracy: 0.3622 - val_loss: 1.0960
Epoch 3/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.3677 - loss: 1.0931 - val_accuracy: 0.3702 - val_loss: 1.0962
Epoch 4/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.3623 - loss: 1.0939 - val_accuracy: 0.3378 - val_loss: 1.0939
Epoch 5/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.3686 - loss: 1.0921 - val_accuracy: 0.3629 - val_loss: 1.0966
Epoch 6/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.3658 - loss: 1.0948 - val_accuracy: 0.3664 - val_loss: 1.0949
Epoch 7/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.3665 - loss: 1.0942 - val_accuracy: 0.3632 - val_loss: 1.0974
Epoch 8/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.3663 - loss: 1.0946 - 

In [9]:
from tensorflow.keras.layers import Input,Dense,Embedding,SimpleRNN
from tensorflow.keras.models import Sequential

#set vocab size
vocab_size=len(tokenizer.word_index)+1
#vocab_size
max_token_id = int(max(np.max(x_train), np.max(x_test)))
vocab_size = max_token_id + 1

print(f"--- VOCAB CHECK ---")
print(f"Tokenizer unique words: {len(tokenizer.word_index)}")
print(f"Highest Token ID present in data arrays: {max_token_id}")
print(f"Setting Embedding input_dim to: {vocab_size}")
print(f"-------------------")

#create model with embedding layer
model_final=Sequential([Embedding(input_dim=vocab_size, output_dim=128,input_length=99, mask_zero=True), SimpleRNN(32),Dense(3,activation='softmax') ])

#compile model
model_final.compile(optimizer='adam',metrics=['accuracy'],loss='sparse_categorical_crossentropy')
model_final.summary()

#fit model
model_final.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=10,batch_size=64)


--- VOCAB CHECK ---
Tokenizer unique words: 29229
Highest Token ID present in data arrays: 29229
Setting Embedding input_dim to: 29230
-------------------


c:\Users\91976\Desktop\Generative AI Course\venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 68s 90ms/step - accuracy: 0.7609 - loss: 0.5805 - val_accuracy: 0.8843 - val_loss: 0.3142
Epoch 2/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 60s 84ms/step - accuracy: 0.9510 - loss: 0.1423 - val_accuracy: 0.9086 - val_loss: 0.2537
Epoch 3/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 61s 85ms/step - accuracy: 0.9789 - loss: 0.0623 - val_accuracy: 0.9102 - val_loss: 0.2687
Epoch 4/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 62s 86ms/step - accuracy: 0.9862 - loss: 0.0393 - val_accuracy: 0.8994 - val_loss: 0.3376
Epoch 5/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 62s 87ms/step - accuracy: 0.9862 - loss: 0.0380 - val_accuracy: 0.8986 - val_loss: 0.3463
Epoch 6/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 62s 86ms/step - accuracy: 0.9868 - loss: 0.0355 - val_accuracy: 0.8995 - val_loss: 0.3598
Epoch 7/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 66s 92ms/step - accuracy: 0.9892 - loss: 0.0274 - val_accuracy: 0.9007 - val_loss: 0.3631
Epoch 8/10
719/719 ━━━━━━━━━━━━━━━━━━━━ 65s 90ms/step - accuracy: 0.9902 - loss: 0.0260 - 

In [11]:
#create file 
import pickle

with open('tokenizer.pkl','wb') as file:
    pickle.dump(tokenizer,file)

model_final.save('model.h5')